# Cycle 1 — The Fallen User

> An AI maths tutor divided by zero and hijacked your network. To take it back, prove you know your statistics.

**Your job:** work out three secret numbers — `p`, `q` and `e` — then use them to unlock a hidden message.

The numbers are **not** written in the data. You have to *calculate* each one. (Searching won't help — there is nothing to copy.)

*This first notebook is fully solved for you. Read it, run it, and learn the pattern — you'll do Cycle 2 and 3 yourself.*

## Setup — run these first

Install the libraries, load the data, and import the two helper tools.

In [ ]:
!pip install pandas sympy

In [ ]:
import pandas as pd
from tutor_tools import decrypt_message, sha256_hex

df = pd.read_csv('network_logs.csv')
print('Rows:', len(df))
df.head()

## Step 1 — Find the victim

The victim is the user with the **most failed connections**.

- Keep the rows where `action` is `'connection_fail'`.
- Find the `user_id` that appears most often (the **mode**).

In [ ]:
fails = df[df['action'] == 'connection_fail']
victim = int(fails['user_id'].mode().iloc[0])
print('Victim user_id:', victim)

## Step 2 — Work out `p`

**`p` = the median of the victim's `bytes_up`.**

In [ ]:
victim_rows = df[df['user_id'] == victim]
p = int(victim_rows['bytes_up'].median())
print('p =', p)

## Step 3 — Work out `q`

**`q` = the mode of `bytes_up` among the successful connections** (`action == 'connection_success'`).

In [ ]:
success = df[df['action'] == 'connection_success']
q = int(success['bytes_up'].mode().iloc[0])
print('q =', q)

## Step 4 — Work out `e`

**`e` = how many different `node` values the victim used** (use `.nunique()`).

In [ ]:
e = int(victim_rows['node'].nunique())
print('e =', e)

## Step 5 — Unlock the message

The hidden message belongs to the user with the **highest `bytes_up` among the successful connections**. Find that user's row whose `payload_info` starts with `'ENC:'`, then decrypt it.

*(Some rows have fake `ENC:` payloads — only the right one decrypts to real text.)*

In [ ]:
carrier = int(success.loc[success['bytes_up'].idxmax(), 'user_id'])
print('Message carrier user_id:', carrier)

enc_row = df[(df['user_id'] == carrier) &
             (df['payload_info'].str.startswith('ENC:', na=False))]
enc_field = enc_row['payload_info'].iloc[0]

message = decrypt_message(enc_field, p, q, e)
print('CYCLE 1 MESSAGE:', message)

## Done!

The message gives you an **IP address**. Write it down — you'll need it to start **Cycle 2**.